# Teacher Analytics — Synthetic Dataset Generator
بيولّد 1000 طالب بكل الجداول (Students, Groups, Sessions, Records, Payments, Meta, Subscribers) + ground truth + نسخة dirty.

**شغّل الـ Cells بالترتيب من 1 لـ 12. Cell 13 اختياري (رفع على Google Sheets).** كل Cell مكتوب فوقه يتحط بعد أنهي واحدة.

### Cell 1 — Config | كل الإعدادات في مكان واحد (ابدأ بالخلية دي)
**يتحط:** أول Cell في الـ notebook

In [ ]:
import math, random, os, json, datetime as dt
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)

N_STUDENTS = 1000
GRADE_GROUPS = {"first": 13, "second": 13, "third": 14}   # = 40 groups
N_GROUPS = sum(GRADE_GROUPS.values())
START_DATE = dt.date(2025, 9, 6)        # Saturday - first possible session
END_LIMIT = dt.date(2026, 7, 4)         # hard upper bound (most groups finish in June)
SESSIONS_PER_GROUP = 36                 # 9 billing months x 4 sessions
TODAY = dt.date(2026, 11, 16)           # no date in the data may be after this
OUT_DIR = "output"
MAKE_DIRTY = True

# payment behaviour weights (before adjustments)
PAY_PROFILES = {
    "on_time": 0.45, "sometimes_late": 0.13, "chronic_late": 0.09,
    "skip_catchup": 0.05, "stopped_paying": 0.08, "partial_payer": 0.08, "one_miss": 0.12,
}
FEE_BASE = {"first": 600, "second": 650, "third": 800}
LANG_EXTRA_FEE = 100

### Cell 2 — Reference data | أسماء وأحياء وأيام وفترات ورسائل
**يتحط:** بعد Cell 1

In [ ]:
MALE = ["أحمد","محمد","محمود","مصطفى","عمر","يوسف","كريم","خالد","علي","حسن","حسين","إبراهيم","عبد الرحمن",
        "ياسين","زياد","مروان","حمزة","بلال","طارق","سيف الدين","حسام","مازن","شريف","عمرو","يحيى","آدم","أنس",
        "فارس","تامر","هشام","وليد","إسلام","أيمن","رامي","سامح","معتز","نور الدين","بدر","جاد","أسامة","عماد",
        "ماجد","هاني","ياسر","صلاح","عادل","مهند","كارم","أدهم","سليم","رضا","عبد الله","معاذ","زين"]
FEMALE = ["سارة","مريم","فاطمة","سلمى","آية","ندى","شهد","حبيبة","ملك","هنا","ريم","فريدة","نورهان","ياسمين",
          "جنة","ليلى","هاجر","دنيا","مي","رنا","هبة","علياء","جودي","روان","نور","رودينا","منة الله","إسراء",
          "بسمة","هدير","ميرنا","لجين","تقى","رهف","يارا","جنى","سما","أمل","دعاء","ضحى","مليكة","روضة"]
FATHERS = ["محمد","أحمد","محمود","مصطفى","علي","حسن","حسين","إبراهيم","خالد","عمرو","سعيد","فاروق","رفعت","منصور",
           "نبيل","شاكر","صابر","رشاد","ماهر","حامد","فؤاد","سمير","كمال","مجدي","عثمان","عبد الله","عبد العزيز",
           "عبد الحميد","جمال","ممدوح","أنور","سيد","رمضان","عبد الفتاح","إمام","عوض","بكر","زكريا","طارق","شريف",
           "عادل","هشام","وليد","ياسر","صلاح","عماد","سامي","مدحت","رأفت"]
AREAS = [("سموحة",9),("ميامي",7),("سيدي جابر",8),("العصافرة",6),("المنتزه",5),("العجمي",5),("الإبراهيمية",6),
         ("كفر عبده",4),("رشدي",5),("سان ستيفانو",4),("الورديان",3),("محرم بك",4),("السيوف",6),("فلمنج",4),
         ("كليوباترا",4),("الشاطبي",3),("المندرة",3),("سيدي بشر",6),("لوران",4),("الأزاريطة",2)]
AR_DAY = {5: "السبت", 6: "الأحد", 0: "الاثنين", 1: "الثلاثاء", 2: "الأربعاء", 3: "الخميس", 4: "الجمعة"}
SLOTS = [(wd, h) for wd in [5, 6, 0, 1, 2, 3] for h in [14, 16, 18, 20]] + [(5, 10), (3, 10)] + [(4, h) for h in [10, 12, 14, 16]]
YEAR_WORDS = {"first": ["أولى", "اولى"], "second": ["تانية", "ثانية"], "third": ["تالتة", "ثالثة"]}
GRADE_LABEL = {"first": "أولى ثانوي", "second": "تانية ثانوي", "third": "تالتة ثانوي"}

ABSENT_REASONS_EXCUSED = ["مريض", "ظرف عائلي", "سفر", "امتحانات مدرسة"]
ABSENT_REASONS_UNEXCUSED = ["نسي الميعاد", "بدون عذر", "بدون عذر", "امتحانات مدرسة", "تعب من المدرسة"]
NOTES_CUSTOM_HIGH = ["تحسن ملحوظ", "طالب متميز", "مشاركة ممتازة في الحصة", "حل كل التمارين صح"]
NOTES_CUSTOM_MID = ["محتاج مراجعة على الدرس السابق", "كثير الكلام في الحصة", "لازم يركز أكتر", "مستواه مستقر"]
NOTES_CUSTOM_LOW = ["لازم ولي الأمر يتواصل", "محتاج خطة علاج", "ضعف واضح في الأساسيات", "ملاحظ إنه مش مذاكر"]
PAY_METHODS = [("كاش", 0.68), ("إنستاباي", 0.16), ("فودافون كاش", 0.11), ("تحويل بنكي", 0.05)]


def clip(x, a=0.0, b=1.0):
    return max(a, min(b, x))


def wchoice(pairs):
    items, w = zip(*pairs)
    w = np.array(w, dtype=float)
    return items[int(rng.choice(len(items), p=w / w.sum()))]


def hour12(h):
    return h if h <= 12 else h - 12


def time_str(h):
    return f"{hour12(h)}:00:00 {'AM' if h < 12 else 'PM'}"

### Cell 3 — Groups | 40 مجموعة (20-30 طالب) بأسماء غير مرتبة وبتناقضات مقصودة
**يتحط:** بعد Cell 2

In [ ]:
grades = ["first"] * GRADE_GROUPS["first"] + ["second"] * GRADE_GROUPS["second"] + ["third"] * GRADE_GROUPS["third"]
rng.shuffle(grades)
lang_idx = set(rng.choice(N_GROUPS, size=6, replace=False).tolist())

sizes = rng.integers(20, 31, size=N_GROUPS)
while sizes.sum() != N_STUDENTS:
    i = int(rng.integers(0, N_GROUPS))
    if sizes.sum() > N_STUDENTS and sizes[i] > 20:
        sizes[i] -= 1
    elif sizes.sum() < N_STUDENTS and sizes[i] < 30:
        sizes[i] += 1

slot_count = Counter(); slot_grades = defaultdict(set)
groups = []
strong = set(rng.choice(N_GROUPS, size=2, replace=False).tolist())
weak = set(rng.choice([i for i in range(N_GROUPS) if i not in strong], size=2, replace=False).tolist())
weak_att = int(rng.choice([i for i in range(N_GROUPS) if i not in strong | weak]))
used_names = set()
for i in range(N_GROUPS):
    grade = grades[i]
    for _ in range(500):
        wd, h = SLOTS[int(rng.integers(0, len(SLOTS)))]
        if slot_count[(wd, h)] < 2 and grade not in slot_grades[(wd, h)]:
            break
    slot_count[(wd, h)] += 1; slot_grades[(wd, h)].add(grade)
    track = "لغات" if i in lang_idx else "عام"
    yw = YEAR_WORDS[grade][0 if rng.random() < 0.7 else 1]
    prefix = "" if rng.random() < 0.15 else "مجموعة "
    mid = f"لغات {yw}" if track == "لغات" else yw
    name = f"{prefix}{mid} {AR_DAY[wd]} {hour12(h)}"
    while name in used_names:
        name += " "
    used_names.add(name)
    groups.append(dict(
        idx=i, size=int(sizes[i]), grade=grade, track=track, wd=wd, hour=h, name=name,
        fee=FEE_BASE[grade] + (LANG_EXTRA_FEE if track == "لغات" else 0),
        quiz_max=20 if rng.random() < 0.8 else 10, month_max=50 if rng.random() < 0.75 else 60,
        lvl_eff=float(rng.normal(0, 0.03)) + (0.05 if i in strong else 0) - (0.05 if i in weak else 0),
        att_eff=float(rng.normal(0, 0.02)) - (0.06 if i == weak_att else 0),
        time_eff=(-0.02 if h >= 20 else 0) + (-0.03 if wd == 4 else 0),
    ))
# group ids are assigned in shuffled order => not sorted by grade
for i, g in enumerate(groups):
    g["group_id"] = f"G{i + 1:02d}"
G = {g["group_id"]: g for g in groups}

### Cell 4 — Sessions | جدول الحصص (حصة أسبوعية + إلغاءات + تعويض + إجازات)
**يتحط:** بعد Cell 3

In [ ]:
SKIP_RANGES = [(dt.date(2026, 1, 24), dt.date(2026, 2, 7)),    # mid-year break
               (dt.date(2026, 3, 19), dt.date(2026, 3, 25)),   # Eid al-Fitr week (approx)
               (dt.date(2026, 5, 24), dt.date(2026, 5, 31))]   # Eid al-Adha week (approx)
SKIP_DATES = {dt.date(2025, 10, 6), dt.date(2026, 1, 7), dt.date(2026, 4, 13), dt.date(2026, 5, 1)}


def is_skipped(d):
    return d in SKIP_DATES or any(a <= d <= b for a, b in SKIP_RANGES)


def first_on_or_after(d0, wd):
    return d0 + dt.timedelta(days=(wd - d0.weekday()) % 7)


sessions = []
group_sessions = {}
for g in groups:
    d = first_on_or_after(START_DATE, g["wd"])
    n = 0; cancels = int(rng.integers(2, 5)); makeups = int(rng.integers(1, 3)); rows = []
    while n < SESSIONS_PER_GROUP and d <= END_LIMIT:
        if is_skipped(d):
            d += dt.timedelta(days=7); continue
        if cancels > 0 and n > 2 and rng.random() < 0.08:
            cancels -= 1
            if makeups > 0 and rng.random() < 0.6:
                md = d + dt.timedelta(days=int(rng.integers(1, 4)))
                if not is_skipped(md) and md <= END_LIMIT:
                    makeups -= 1; n += 1
                    rows.append(dict(group_id=g["group_id"], date=md, session_number=n, month=math.ceil(n / 4), session_type="makeup"))
            d += dt.timedelta(days=7); continue
        n += 1
        rows.append(dict(group_id=g["group_id"], date=d, session_number=n, month=math.ceil(n / 4), session_type="regular"))
        d += dt.timedelta(days=7)
    group_sessions[g["group_id"]] = rows
sid_counter = 0
for gid in sorted(group_sessions):
    for r in group_sessions[gid]:
        sid_counter += 1
        r["session_id"] = f"SES{sid_counter:04d}"
        sessions.append(r)
for g in groups:
    g["first_date"] = group_sessions[g["group_id"]][0]["date"]

### Cell 5 — Students | 1000 طالب + أسماء + أخوات + شخصيات خفية (archetypes)
**يتحط:** بعد Cell 4

In [ ]:
ARCH_COUNTS = {"star": 130, "rising": 120, "average": 490, "volatile": 30, "gifted_absent": 15, "hardworker_low": 25,
               "declining": 25, "weak": 40, "at_risk": 60, "needs_attention": 35, "dropout": 30}
ARCH = {
    "star": dict(L0=(0.92, 0.03), slope=0.0, sigma=0.04, att=(0.98, 0.01), hw=(0.92, 0.04)),
    "rising": dict(L0=(0.79, 0.05), slope=0.025, sigma=0.07, att=(0.94, 0.025), hw=(0.78, 0.08)),
    "average": dict(L0=(0.89, 0.04), slope=0.0, sigma=0.08, att=(0.965, 0.015), hw=(0.78, 0.10)),
    "volatile": dict(L0=(0.74, 0.06), slope=0.0, sigma=0.18, att=(0.90, 0.04), hw=(0.60, 0.12)),
    "gifted_absent": dict(L0=(0.88, 0.04), slope=0.0, sigma=0.07, att=(0.68, 0.05), hw=(0.55, 0.10)),
    "hardworker_low": dict(L0=(0.52, 0.05), slope=0.01, sigma=0.09, att=(0.97, 0.02), hw=(0.90, 0.05)),
    "declining": dict(L0=(0.88, 0.04), slope=0.0, sigma=0.08, att=(0.95, 0.02), hw=(0.85, 0.05)),
    "weak": dict(L0=(0.46, 0.06), slope=0.0, sigma=0.10, att=(0.91, 0.04), hw=(0.45, 0.10)),
    "at_risk": dict(L0=(0.55, 0.05), slope=-0.005, sigma=0.10, att=(0.70, 0.05), hw=(0.40, 0.10)),
    "needs_attention_A": dict(L0=(0.80, 0.04), slope=0.0, sigma=0.08, att=(0.80, 0.02), hw=(0.65, 0.10)),
    "needs_attention_B": dict(L0=(0.66, 0.04), slope=0.0, sigma=0.08, att=(0.93, 0.02), hw=(0.60, 0.10)),
    "dropout": dict(L0=(0.62, 0.07), slope=-0.01, sigma=0.10, att=(0.86, 0.04), hw=(0.50, 0.10)),
}
arch_list = [a for a, c in ARCH_COUNTS.items() for _ in range(c)]
rng.shuffle(arch_list)

students = []
used_full = set(); used_phone = set()


def new_phone():
    while True:
        p = "01" + str(rng.choice([0, 1, 2, 5])) + "".join(str(int(x)) for x in rng.integers(0, 10, size=8))
        if p not in used_phone:
            used_phone.add(p); return p


sid = 0
for gid in sorted(G):
    g = G[gid]
    for _ in range(g["size"]):
        sid += 1
        gender = "ذكر" if rng.random() < 0.5 else "أنثى"
        while True:
            first = random.choice(MALE if gender == "ذكر" else FEMALE)
            father = random.choice(FATHERS); gf = random.choice(FATHERS)
            full = f"{first} {father} {gf}"
            if full not in used_full:
                used_full.add(full); break
        school = wchoice([("لغات", 0.85), ("خاص", 0.10), ("حكومي", 0.05)]) if g["track"] == "لغات" else \
            wchoice([("حكومي", 0.65), ("خاص", 0.20), ("لغات", 0.15)])
        students.append(dict(student_id=f"S{sid:04d}", n=sid, gender=gender, first=first, father=father, gf=gf,
                             phone=new_phone(), parent_phone=new_phone(), group_id=gid, grade=g["grade"],
                             school_type=school, area=wchoice(AREAS), archetype=arch_list[sid - 1],
                             origin=None, transfer=False, transfer_date=None, late_joiner=False, long_excused=False,
                             sibling_of=None, name_collision=None, status="active", leave_date=None, shock=False,
                             discount_type="none", discount=0.0))
S = {s["student_id"]: s for s in students}
by_group = defaultdict(list)
for s in students:
    by_group[s["group_id"]].append(s)

# siblings (~8% of students: 40 pairs, different groups, share parent phone + family names)
free = set(S)
pairs = 0
while pairs < 40:
    a, b = random.sample(sorted(free), 2)
    if S[a]["group_id"] == S[b]["group_id"]:
        continue
    free -= {a, b}
    sa, sb = S[a], S[b]
    used_full.discard(f"{sb['first']} {sb['father']} {sb['gf']}")
    sb["father"], sb["gf"], sb["parent_phone"] = sa["father"], sa["gf"], sa["parent_phone"]
    if f"{sb['first']} {sb['father']} {sb['gf']}" in used_full or sb["first"] == sa["first"]:
        sb["first"] = random.choice(MALE if sb["gender"] == "ذكر" else FEMALE)
    used_full.add(f"{sb['first']} {sb['father']} {sb['gf']}")
    sb["sibling_of"] = a; sa["sibling_of"] = sa["sibling_of"] or "elder"
    sb["discount_type"], sb["discount"] = "sibling", 0.20
    pairs += 1

# name collisions: 6 pairs share first+father only; 2 pairs are exact same full name (different groups)
cands = [s for s in students if s["sibling_of"] is None]
for k in range(8):
    while True:
        a, b = random.sample(cands, 2)
        if a["gender"] == b["gender"] and a["group_id"] != b["group_id"] and a["name_collision"] is None and b["name_collision"] is None:
            break
    b["first"], b["father"] = a["first"], a["father"]
    if k < 2:
        b["gf"] = a["gf"]
    a["name_collision"] = b["name_collision"] = "exact" if k < 2 else "first+father"
for s in students:
    s["name"] = f"{s['first']} {s['father']} {s['gf']}"

# overlays
ids = [s["student_id"] for s in students]
non_drop = [i for i in ids if S[i]["archetype"] != "dropout"]
for i in rng.choice(non_drop, size=20, replace=False):
    S[i]["long_excused"] = True
late_pool = [i for i in non_drop]
for i in rng.choice(late_pool, size=50, replace=False):
    S[i]["late_joiner"] = True
    S[i]["join_cycle"] = int(rng.integers(2, 7))
trans_pool = [i for i in non_drop if not S[i]["late_joiner"]]
rng.shuffle(trans_pool)
outbound = Counter(); tcount = 0
for i in trans_pool:
    if tcount >= 30:
        break
    s = S[i]
    opts = [g["group_id"] for g in groups if g["grade"] == s["grade"] and g["group_id"] != s["group_id"]
            and g["size"] <= 28 and outbound[g["group_id"]] < 2]
    if not opts:
        continue
    o = random.choice(opts); outbound[o] += 1; tcount += 1
    t = int(rng.integers(3, 7))
    s.update(transfer=True, origin=o, transfer_cycle=t,
             transfer_date=group_sessions[s["group_id"]][4 * t]["date"])
for i in rng.choice(ids, size=60, replace=False):
    S[i]["shock"] = True

# discounts / exemptions
pool = [i for i in ids if S[i]["discount_type"] == "none"]
for i in rng.choice(pool, size=30, replace=False):
    S[i]["discount_type"], S[i]["discount"] = "exempt", 1.0
stars = [i for i in ids if S[i]["archetype"] == "star" and S[i]["discount_type"] == "none"]
for i in rng.choice(stars, size=20, replace=False):
    S[i]["discount_type"], S[i]["discount"] = "scholarship", 0.5

# latent parameters
for s in students:
    key = s["archetype"]
    if key == "needs_attention":
        key = "needs_attention_A" if rng.random() < 0.5 else "needs_attention_B"
    p = ARCH[key]
    s["L0"] = clip(float(rng.normal(*p["L0"])), 0.25, 0.99)
    s["slope"] = p["slope"]; s["sigma"] = p["sigma"]
    s["att"] = clip(float(rng.normal(*p["att"])), 0.45, 0.995)
    s["hw"] = clip(float(rng.normal(*p["hw"])), 0.1, 0.98)
    s["late_p"] = clip(float(rng.normal(0.02, 0.015)), 0.0, 0.10)
    s["family_stability"] = float(rng.beta(5, 2))
    # enrollment / dropout dates
    gs = group_sessions[s["group_id"]]
    if s["late_joiner"]:
        s["join_date"] = gs[4 * (s["join_cycle"] - 1)]["date"] - dt.timedelta(days=int(rng.integers(0, 4)))
    else:
        s["join_date"] = dt.date(2025, 8, 25) + dt.timedelta(days=int(rng.integers(0, 12)))
    if s["archetype"] == "dropout":
        c = int(rng.integers(3, 9)); pos = int(rng.integers(0, 4))
        J = min(4 * (c - 1) + pos, len(gs) - 1)
        s.update(status="dropped", leave_date=gs[J]["date"], drop_cycle=c, decay_k=int(rng.integers(4, 8)))
    else:
        s["decay_k"] = 0

### Cell 6 — Records | الحضور والواجب والدرجات (محاكاة بأسباب مخفية + ضوضاء)
**يتحط:** بعد Cell 5

In [ ]:
def seas_att(d, grade):
    s = 0.0
    if dt.date(2026, 1, 3) <= d <= dt.date(2026, 1, 22): s -= 0.05
    if dt.date(2026, 2, 18) <= d <= dt.date(2026, 3, 18): s -= 0.03
    if grade == "third" and d >= dt.date(2026, 4, 15): s += 0.03
    if d <= dt.date(2025, 9, 20): s += 0.02
    return s


def seas_lvl(d, grade):
    s = 0.0
    if d.month == 9 and d.year == 2025: s -= 0.03
    if dt.date(2026, 1, 3) <= d <= dt.date(2026, 1, 22): s -= 0.02
    if grade == "third" and d >= dt.date(2026, 4, 1): s += 0.03
    return s


def hw_label(q):
    return "ممتاز" if q >= 0.88 else "جيد جدا" if q >= 0.76 else "جيد" if q >= 0.62 else "مقبول" if q >= 0.48 else "ضعيف"


def note_for(p):
    return ("ممتاز جدا" if p >= 0.95 else "أداء جيد جدا" if p >= 0.85 else "أداء جيد" if p >= 0.75
            else "مستوى متوسط" if p >= 0.60 else "يحتاج متابعة" if p >= 0.50 else "ضعيف")


def student_segments(s):
    if s["transfer"]:
        T = s["transfer_date"]
        segs = [(s["origin"], x) for x in group_sessions[s["origin"]] if s["join_date"] <= x["date"] < T]
        segs += [(s["group_id"], x) for x in group_sessions[s["group_id"]] if x["date"] >= T]
    else:
        segs = [(s["group_id"], x) for x in group_sessions[s["group_id"]] if x["date"] >= s["join_date"]]
    if s["leave_date"]:
        segs = [(g, x) for g, x in segs if x["date"] < s["leave_date"]]
    return segs


records = []
for s in students:
    segs = student_segments(s); n = len(segs); K = s["decay_k"]
    shock_i = int(rng.integers(4, max(5, n))) if s["shock"] else -1
    if s["long_excused"]:
        bs = int(rng.integers(6, max(7, n - 8))); bl = int(rng.integers(2, 5)); blk_reason = random.choice(["مريض", "سفر"])
    else:
        bs = -10; bl = 0
    prev_abs = False
    for i, (gid, x) in enumerate(segs):
        gi = G[gid]; m = x["month"]; d = x["date"]; rem = n - i
        lvl = s["L0"] + s["slope"] * (m - 1) + gi["lvl_eff"] + seas_lvl(d, gi["grade"])
        if s["archetype"] == "declining": lvl -= 0.07 * max(0, m - 3)
        p = s["att"] + gi["att_eff"] + gi["time_eff"] + seas_att(d, gi["grade"])
        if s["archetype"] == "declining": p -= 0.008 * max(0, m - 3)
        if K and rem <= K: p *= 0.3 + 0.7 * (rem - 1) / K
        a = clip(1 - p, 0.01, 0.92)
        pa = a + 0.25 * (1 - a) if prev_abs else a * 0.75
        if bs <= i < bs + bl:
            att, reason = "excused", blk_reason; prev_abs = True
        elif rng.random() < pa:
            if rng.random() < 0.35:
                att, reason = "excused", random.choice(ABSENT_REASONS_EXCUSED)
            else:
                att, reason = "absent", random.choice(ABSENT_REASONS_UNEXCUSED)
            prev_abs = True
        else:
            prev_abs = False
            if rng.random() < s["late_p"]:
                att = "late"; reason = "تأخر عن الميعاد" if rng.random() < 0.4 else ""
            else:
                att, reason = "present", ""
        # homework
        hws = s["hw"]
        if s["archetype"] == "declining": hws -= 0.06 * max(0, m - 1)
        if K and rem <= 8: hws -= 0.12
        hws = clip(hws, 0.05, 0.98)
        q = 0.55 * hws + 0.45 * clip(lvl) + float(rng.normal(0, 0.12))
        no_sub = clip((1 - hws) * 0.4, 0.02, 0.7) + (0.15 if (K and rem <= 6) else 0)
        if att in ("absent", "excused"):
            homework = "لم يسلم" if rng.random() < 0.7 else hw_label(q)
        else:
            homework = "لم يسلم" if rng.random() < no_sub else hw_label(q)
        # scores
        is_month_exam = x["session_number"] % 4 == 0
        mx = gi["month_max"] if is_month_exam else gi["quiz_max"]
        if att in ("present", "late"):
            noise = float(rng.normal(0, s["sigma"] * (1.15 if is_month_exam else 1.0)))
            pct = lvl + noise - (0.03 if is_month_exam else 0) - (0.03 if att == "late" else 0)
            if i == shock_i: pct -= 0.35
            score = int(round(clip(pct) * mx))
            percentage = score / mx
            note = note_for(percentage)
            r = rng.random()
            if r < 0.10:
                note = random.choice(NOTES_CUSTOM_HIGH if percentage >= 0.85 else NOTES_CUSTOM_MID if percentage >= 0.6 else NOTES_CUSTOM_LOW)
            exam_score, max_score = score, mx
        else:
            exam_score = max_score = percentage = None; note = ""
        records.append(dict(student_id=s["student_id"], session_id=x["session_id"], group_id=gid, date=d,
                            attendance=att, homework=homework, exam_score=exam_score, max_exam_score=max_score,
                            percentage=percentage, notes=note, group_month=m, group_session=x["session_number"],
                            attendance_notes=reason))

rec = pd.DataFrame(records).sort_values(["session_id", "student_id"]).reset_index(drop=True)
rec.insert(0, "record_id", [f"R{i + 1:05d}" for i in range(len(rec))])
rec = rec.sort_values(["student_id", "date", "session_id"])
rec["student_session_number"] = rec.groupby("student_id").cumcount() + 1
rec["student_month"] = np.ceil(rec["student_session_number"] / 4).astype(int)
rec["student_session"] = (rec["student_session_number"] - 1) % 4 + 1
rec = rec.sort_values("record_id").reset_index(drop=True)

### Cell 7 — Payments | المدفوعات (دفع / متأخر / جزئي / متوقف / معفي) بشكل مرتبط بالأداء
**يتحط:** بعد Cell 6

In [ ]:
mon_by_student = rec.groupby("student_id").apply(lambda d: sorted(d["group_month"].unique())).to_dict()
grp_month_first = {gid: {} for gid in G}
for gid, rows in group_sessions.items():
    for r in rows:
        grp_month_first[gid].setdefault(r["month"], r["date"])
stud_month_group = rec.groupby(["student_id", "group_month"])["group_id"].last().to_dict()
problem = [p for p in PAY_PROFILES if p != "on_time"]
pay_rows = []
for s in students:
    months = mon_by_student.get(s["student_id"], [])
    if not months:
        continue
    f = s["family_stability"]
    mult = 0.5 + 2.0 * (1 - f)
    if s["archetype"] in ("at_risk", "weak", "dropout", "declining"): mult *= 1.4
    w = {k: v * (mult if k in problem else 1.0) for k, v in PAY_PROFILES.items()}
    tot = sum(w.values())
    prof = rng.choice(list(w), p=[w[k] / tot for k in w])
    s["pay_profile"] = "exempt" if s["discount_type"] == "exempt" else str(prof)
    due_amt = round(G[s["group_id"]]["fee"] * (1 - s["discount"]) / 10) * 10
    k_stop = int(rng.integers(2, 6)); k_skip = int(rng.integers(3, 8)); k_miss = int(rng.integers(2, 9))
    for m in months:
        gid = stud_month_group[(s["student_id"], m)]
        due_date = grp_month_first[gid][m]
        amount_due = round(G[gid]["fee"] * (1 - s["discount"]) / 10) * 10
        status, paid_amt, paid_date, method = "paid", amount_due, None, None
        pf = s["pay_profile"]
        if pf == "exempt":
            status, amount_due, paid_amt = "exempt", 0, 0
        elif pf == "on_time":
            paid_date = due_date + dt.timedelta(days=int(rng.integers(-4, 4)))
        elif pf == "sometimes_late":
            if rng.random() < 0.25: status = "paid_late"
        elif pf == "chronic_late":
            r = rng.random()
            status = "paid_late" if r < 0.65 else ("unpaid" if r < 0.75 else "paid")
        elif pf == "skip_catchup":
            if m == k_skip: status = "paid_late"
        elif pf == "stopped_paying":
            if m > k_stop: status = "unpaid"
        elif pf == "partial_payer":
            if rng.random() < 0.45: status = "partial"
        elif pf == "one_miss":
            if m == k_miss: status = "unpaid"
        # dropout lapses before leaving
        if s["status"] == "dropped" and pf != "exempt":
            if m == months[-1] and rng.random() < 0.65: status = "unpaid"
            elif len(months) > 1 and m == months[-2] and rng.random() < 0.4 and status == "paid": status = "paid_late"
        if status == "paid" and paid_date is None:
            paid_date = due_date + dt.timedelta(days=int(rng.integers(-4, 4)))
        elif status == "paid_late":
            if pf == "skip_catchup":
                nxt = grp_month_first[gid].get(m + 1, due_date + dt.timedelta(days=28))
                paid_date = nxt + dt.timedelta(days=int(rng.integers(0, 4)))
            else:
                paid_date = due_date + dt.timedelta(days=int(rng.integers(5, 29)))
        elif status == "partial":
            paid_amt = int(round(amount_due * float(rng.uniform(0.4, 0.8)) / 10) * 10)
            paid_date = due_date + dt.timedelta(days=int(rng.integers(3, 25)))
        elif status == "unpaid":
            paid_amt = 0; paid_date = None
        if paid_date is not None and paid_date > TODAY: paid_date = TODAY
        if status in ("paid", "paid_late", "partial"):
            method = wchoice(PAY_METHODS)
        pay_rows.append(dict(student_id=s["student_id"], month=m, month_label=f"شهر {m}", status=status,
                             amount_due=amount_due, amount_paid=paid_amt, due_date=due_date, paid_date=paid_date, method=method))
pay = pd.DataFrame(pay_rows).sort_values(["month", "student_id"]).reset_index(drop=True)
pay.insert(0, "payment_id", [f"P{i + 1:05d}" for i in range(len(pay))])

### Cell 8 — Final tables | تجميع الجداول بنفس أسماء وترتيب أعمدة الشيت الأصلي (الأعمدة الإضافية يمين)
**يتحط:** بعد Cell 7

In [ ]:
T_students = pd.DataFrame([dict(student_id=s["student_id"], name=s["name"], phone=s["phone"], parent_phone=s["parent_phone"],
                                group_id=s["group_id"], gender=s["gender"], grade=GRADE_LABEL[s["grade"]],
                                school_type=s["school_type"], area=s["area"], join_date=s["join_date"],
                                status=s["status"], leave_date=s["leave_date"]) for s in students])
T_groups = pd.DataFrame([dict(group_id=g["group_id"], group_name=g["name"], day=AR_DAY[g["wd"]], time=time_str(g["hour"]),
                              grade=GRADE_LABEL[g["grade"]], track=g["track"], monthly_fee=g["fee"], start_date=g["first_date"])
                         for g in groups])
T_sessions = pd.DataFrame(sessions)[["session_id", "group_id", "date", "session_number", "month", "session_type"]]
T_records = rec[["record_id", "student_id", "session_id", "date", "attendance", "homework", "exam_score", "max_exam_score",
                 "percentage", "notes", "student_session_number", "group_month", "group_session", "student_month",
                 "student_session", "attendance_notes"]].copy()
T_payments = pay[["payment_id", "student_id", "month_label", "status", "amount_due", "amount_paid", "due_date", "paid_date", "method"]].copy()
for _c in ("exam_score", "max_exam_score"):
    T_records[_c] = T_records[_c].astype("Int64")      # clean integers in CSV/Sheets (blank when absent)

# Subscribers (telegram links) + Meta
sub_rows = []
sib_chat = {}
for s in students:
    first_date = G[s["group_id"]]["first_date"]
    def ts():
        t = dt.datetime.combine(first_date, dt.time(0)) + dt.timedelta(seconds=int(rng.integers(0, 60 * 86400)))
        return t.strftime("%Y-%m-%dT%H:%M:%S.") + f"{int(rng.integers(0, 1000)):03d}Z"
    if rng.random() < 0.35:
        c = int(rng.integers(1_000_000_000, 7_000_000_000))
        sub_rows.append(dict(key=f"{c}_{s['student_id']}", chat_id=str(c), student_id=s["student_id"], role="student",
                             notify="TRUE" if rng.random() < 0.6 else "", verified_at=ts()))
    if rng.random() < 0.28:
        pk = s["parent_phone"]
        c = sib_chat.setdefault(pk, int(rng.integers(1_000_000_000, 7_000_000_000)))
        sub_rows.append(dict(key=f"{c}_{s['student_id']}", chat_id=str(c), student_id=s["student_id"], role="parent",
                             notify="TRUE" if rng.random() < 0.85 else "FALSE", verified_at=ts()))
T_subs = pd.DataFrame(sub_rows)
T_meta = pd.DataFrame([("last_student_id", T_students.student_id.iloc[-1]), ("last_group_id", T_groups.group_id.iloc[-1]),
                       ("last_session_id", T_sessions.session_id.iloc[-1]), ("last_record_id", T_records.record_id.iloc[-1]),
                       ("last_payment_id", T_payments.payment_id.iloc[-1])], columns=["key", "value"])

### Cell 9 — Validation + calibration | فحوصات PASS/FAIL وجدول المعايرة
**يتحط:** بعد Cell 8

In [ ]:
checks = []


def check(name, cond):
    checks.append((name, bool(cond)))


check("1000 unique students", T_students.student_id.is_unique and len(T_students) == 1000)
check("group count = 40", len(T_groups) == 40)
gs_sizes = T_students.groupby("group_id").size()
check("group sizes 20-30", gs_sizes.between(20, 30).all())
for t, c in [("students", "student_id"), ("groups", "group_id"), ("sessions", "session_id"), ("records", "record_id"), ("payments", "payment_id")]:
    check(f"unique {c}", globals()[f"T_{t}"][c].is_unique)
check("FK records.student", T_records.student_id.isin(T_students.student_id).all())
check("FK records.session", T_records.session_id.isin(T_sessions.session_id).all())
check("FK sessions.group", T_sessions.group_id.isin(T_groups.group_id).all())
check("FK students.group", T_students.group_id.isin(T_groups.group_id).all())
check("FK payments.student", T_payments.student_id.isin(T_students.student_id).all())
check("no orphan sessions (every session has records)", T_sessions.session_id.isin(T_records.session_id).all())
wd_ok = T_sessions[T_sessions.session_type == "regular"].merge(T_groups[["group_id", "day"]], on="group_id")
check("regular session weekday == group.day", (wd_ok.date.map(lambda d: AR_DAY[d.weekday()]) == wd_ok.day).all())
check("month = ceil(session_number/4)", (T_sessions.month == np.ceil(T_sessions.session_number / 4)).all())
check("sessions per group == 36", (T_sessions.groupby("group_id").size() == SESSIONS_PER_GROUP).all())
check("no duplicate (group, date) sessions", not T_sessions.duplicated(["group_id", "date"]).any())
pr = T_records.dropna(subset=["percentage"])
check("percentage == score/max", np.allclose(pr.percentage, pr.exam_score / pr.max_exam_score))
check("absent/excused -> blank score/percentage", T_records[T_records.attendance.isin(["absent", "excused"])].percentage.isna().all())
check("score within [0,max]", ((pr.exam_score >= 0) & (pr.exam_score <= pr.max_exam_score)).all())
rj = T_records.merge(T_students[["student_id", "join_date", "leave_date"]], on="student_id")
check("no record before join_date", (rj.date >= rj.join_date).all())
check("no record on/after leave_date", rj.leave_date.isna().sum() + (rj.date < rj.leave_date).sum() == len(rj))
check("no future dates", max(T_records.date.max(), T_sessions.date.max(), T_payments.paid_date.dropna().max()) <= TODAY)
exp_pairs = T_records.assign(m=T_records.group_month).groupby("student_id").m.nunique().sum()
check("one payment row per (student x month with records)", len(T_payments) == exp_pairs and not T_payments.duplicated(["student_id", "month_label"]).any())
check("payment status vocabulary", set(T_payments.status) <= {"paid", "paid_late", "partial", "unpaid", "exempt"})
check("phones unique (student)", T_students.phone.is_unique)
check("no blank/duplicate header rows", not (T_records.record_id == "record_id").any() and not (T_payments.student_id == "student_id").any())

att_all = (T_records.attendance.isin(["present", "late"])).mean()
att_present = (T_records.attendance == "present").mean()
st = T_records.groupby("student_id").agg(n=("record_id", "count"), pres=("attendance", lambda x: (x == "present").mean()),
                                        att=("attendance", lambda x: x.isin(["present", "late"]).mean()),
                                        avg=("percentage", "mean"))
risk = ((st.pres < 0.80) & (st.avg < 0.75))
needs = (((st.pres < 0.85) | (st.avg < 0.80)) & ~risk)
coll = T_payments.amount_paid.sum() / T_payments.amount_due.sum()
calib = pd.DataFrame([
    ("attendance (present+late)", f"{att_all:.1%}", "88-93%"),
    ("attendance (present only, dashboard rule)", f"{att_present:.1%}", "-"),
    ("student avg percentage (mean)", f"{st.avg.mean():.1%}", "76-82%"),
    ("student avg percentage (min-max)", f"{st.avg.min():.0%}-{st.avg.max():.0%}", "35-98%"),
    ("at-risk share (present<80% & avg<75%)", f"{risk.mean():.1%}", "8-12%"),
    ("needs-attention share", f"{needs.mean():.1%}", "18-26%"),
    ("pass rate (percentage>=50%)", f"{(pr.percentage >= 0.5).mean():.1%}", "92-96%"),
    ("collection rate (paid/due)", f"{coll:.1%}", "88-92%"),
    ("rows paid / paid_late / partial / unpaid / exempt", " / ".join(str((T_payments.status == k).sum()) for k in ["paid", "paid_late", "partial", "unpaid", "exempt"]), "-"),
], columns=["metric", "actual", "target"])
print(f"records={len(T_records)} sessions={len(T_sessions)} payments={len(T_payments)} subscribers={len(T_subs)}")
for n_, ok in checks:
    print(("PASS " if ok else "FAIL ") + n_)
print(calib.to_string(index=False))

# extra validation: no session ever has more than 30 students
check("max students per session <= 30", T_records.groupby("session_id").size().max() <= 30)
print("max students in a session:", T_records.groupby("session_id").size().max())

### Cell 10 — Ground truth + case catalog | الحقيقة المخفية (برّه الشيت) + أمثلة لكل حالة للاختبار
**يتحط:** بعد Cell 9

In [ ]:
gt = pd.DataFrame([dict(student_id=s["student_id"], archetype=s["archetype"], needs_attention_pattern=None,
                        L0=round(s["L0"], 3), slope=s["slope"], sigma=s["sigma"], base_attendance=round(s["att"], 3),
                        base_homework=round(s["hw"], 3), late_prob=round(s["late_p"], 3),
                        family_stability=round(s["family_stability"], 3), pay_profile=s.get("pay_profile"),
                        discount_type=s["discount_type"], status=s["status"], drop_cycle=s.get("drop_cycle"),
                        leave_date=s["leave_date"], late_joiner=s["late_joiner"], join_cycle=s.get("join_cycle"),
                        transfer=s["transfer"], origin_group=s["origin"], transfer_date=s["transfer_date"],
                        long_excused=s["long_excused"], has_shock=s["shock"], sibling_of=s["sibling_of"],
                        name_collision=s["name_collision"]) for s in students]).drop(columns=["needs_attention_pattern"])

cr = random.Random(SEED)
cases = []


def add_case(case, ids, detail=""):
    ids = list(ids)
    for i in cr.sample(ids, min(6, len(ids))):
        cases.append(dict(case=case, student_id=i, detail=detail, n_matching=len(ids)))


for a in ARCH_COUNTS:
    add_case(f"archetype:{a}", [s["student_id"] for s in students if s["archetype"] == a])
add_case("overlay:long_excused_block", [s["student_id"] for s in students if s["long_excused"]], "2-4 consecutive excused sessions")
add_case("overlay:transfer_between_groups", [s["student_id"] for s in students if s["transfer"]], "records in two groups; Students.group_id = latest")
add_case("overlay:late_joiner", [s["student_id"] for s in students if s["late_joiner"]], "join mid-year, student_month restarts at 1")
add_case("overlay:one_off_bad_exam_shock", [s["student_id"] for s in students if s["shock"]])
add_case("overlay:sibling_pair_second", [s["student_id"] for s in students if s["discount_type"] == "sibling"], "shares parent_phone, 20% discount")
add_case("names:exact_duplicate_full_name", [s["student_id"] for s in students if s["name_collision"] == "exact"], "same full name as another student")
add_case("names:same_first_and_father", [s["student_id"] for s in students if s["name_collision"] == "first+father"])
for p in list(PAY_PROFILES) + ["exempt"]:
    add_case(f"payment_profile:{p}", [s["student_id"] for s in students if s.get("pay_profile") == p])
add_case("discount:scholarship_50pct", [s["student_id"] for s in students if s["discount_type"] == "scholarship"])
add_case("rule:at_risk_(present<80%_and_avg<75%)", risk[risk].index)
add_case("rule:needs_attention", needs[needs].index)
add_case("rule:perfect_attendance_(no_absence_no_late)", st[st.pres == 1.0].index)
unp = T_payments[T_payments.status == "unpaid"].groupby("student_id").size()
add_case("payments:3+_unpaid_months", unp[unp >= 3].index)
alive_unpaid = [i for i in unp[unp >= 3].index if S[i]["status"] == "active"]
add_case("payments:3+_unpaid_months_but_still_attending", alive_unpaid)
last_unpaid = T_payments.sort_values("payment_id").groupby("student_id").tail(1)
add_case("payments:dropped_with_arrears", [i for i in last_unpaid[last_unpaid.status == "unpaid"].student_id if S[i]["status"] == "dropped"])
case_catalog = pd.DataFrame(cases)

### Cell 11 — Dirty version | نسخة "وسخة" لجزء الـ Data Cleaning (مع log لكل مشكلة اتضافت)
**يتحط:** بعد Cell 10

In [ ]:
issue_log = []
dirty = {}
if MAKE_DIRTY:
    drng = np.random.default_rng(SEED + 1)

    def log(table, key, col, issue, old, new):
        issue_log.append(dict(table=table, key=key, column=col, issue=issue, original=old, injected=new))

    def dirty_copy(df, key):
        d = df.copy().astype(object)
        n = len(d)
        # missing values
        for col in [c for c in d.columns if c not in (key,)]:
            for i in drng.choice(n, size=max(1, int(n * 0.005 / 4)), replace=False):
                if not pd.isna(d.at[i, col]):
                    log(name, d.at[i, key], col, "missing_value", d.at[i, col], None); d.at[i, col] = None
        return d

    for name, df, key in [("Students", T_students, "student_id"), ("Groups", T_groups, "group_id"), ("Sessions", T_sessions, "session_id"),
                          ("Records", T_records, "record_id"), ("Payments", T_payments, "payment_id")]:
        d = dirty_copy(df, key)
        n = len(d)
        # trailing spaces / casing on text columns
        txt = {"Students": ["name", "area"], "Groups": ["group_name"], "Records": ["homework", "notes"], "Payments": ["method"], "Sessions": []}[name]
        for col in txt:
            for i in drng.choice(n, size=max(1, int(n * 0.01)), replace=False):
                v = d.at[i, col]
                if isinstance(v, str):
                    log(name, d.at[i, key], col, "trailing_space", v, v + " "); d.at[i, col] = v + " "
        if name == "Records":
            for i in drng.choice(n, size=int(n * 0.012), replace=False):
                v = d.at[i, "attendance"]
                nv = drng.choice([v.capitalize(), v.upper(), v + " "])
                log(name, d.at[i, key], "attendance", "inconsistent_case_or_space", v, nv); d.at[i, "attendance"] = nv
            ok = [i for i in range(n) if d.at[i, "exam_score"] is not None and not pd.isna(d.at[i, "exam_score"])]
            for i in drng.choice(ok, size=10, replace=False):
                v = d.at[i, "exam_score"]; nv = int(d.at[i, "max_exam_score"]) + int(drng.integers(1, 15)) if drng.random() < 0.6 else -int(drng.integers(1, 6))
                log(name, d.at[i, key], "exam_score", "out_of_range_score", v, nv); d.at[i, "exam_score"] = nv
        if name == "Students":
            for i in drng.choice(n, size=int(n * 0.04), replace=False):
                v = d.at[i, "phone"]
                if isinstance(v, str):
                    nv = drng.choice(["+2" + v, v[:4] + " " + v[4:7] + " " + v[7:], v[1:], "'" + v])
                    log(name, d.at[i, key], "phone", "inconsistent_phone_format", v, nv); d.at[i, "phone"] = nv
        if name == "Payments":
            for i in drng.choice(n, size=int(n * 0.01), replace=False):
                v = d.at[i, "status"]
                nv = {"paid": "Paid", "paid_late": "late", "partial": "partially paid", "unpaid": "UNPAID", "exempt": "exempted"}[v]
                log(name, d.at[i, key], "status", "inconsistent_label", v, nv); d.at[i, "status"] = nv
        # duplicate rows (~0.2%)
        dup_idx = drng.choice(n, size=max(1, int(n * 0.002)), replace=False)
        for i in dup_idx:
            log(name, d.at[i, key], "(row)", "duplicate_row", "", "duplicated")
        d = pd.concat([d, d.loc[dup_idx]], ignore_index=True)
        dirty[name] = d
    issue_log_df = pd.DataFrame(issue_log)
    print("dirty issues injected:", issue_log_df.issue.value_counts().to_dict())

### Cell 12 — Export | ملف xlsx بكل التابات + CSV + ground truth + case catalog
**يتحط:** بعد Cell 11

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

os.makedirs(OUT_DIR, exist_ok=True)
for sub in ("csv", "dirty"):
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
TABLES = {"Students": T_students, "Groups": T_groups, "Sessions": T_sessions, "Records": T_records,
          "Payments": T_payments, "Meta": T_meta, "Subscribers": T_subs}
for name, df in TABLES.items():
    df.to_csv(os.path.join(OUT_DIR, "csv", f"{name}.csv"), index=False, encoding="utf-8-sig")
if MAKE_DIRTY:
    for name, df in dirty.items():
        df.to_csv(os.path.join(OUT_DIR, "dirty", f"{name}_dirty.csv"), index=False, encoding="utf-8-sig")
    issue_log_df.to_csv(os.path.join(OUT_DIR, "dirty", "dirty_issue_log.csv"), index=False, encoding="utf-8-sig")
gt.to_csv(os.path.join(OUT_DIR, "ground_truth_students.csv"), index=False, encoding="utf-8-sig")
case_catalog.to_csv(os.path.join(OUT_DIR, "case_catalog.csv"), index=False, encoding="utf-8-sig")

README = [
    ("Students Management - ML Dataset (synthetic)", ""),
    ("Seed", SEED), ("Students / Groups / Sessions / Records / Payments", f"{len(T_students)} / {len(T_groups)} / {len(T_sessions)} / {len(T_records)} / {len(T_payments)}"),
    ("Academic year (default)", f"{T_sessions.date.min()} -> {T_sessions.date.max()}  (36 sessions = 9 billing months per group)"),
    ("Original columns", "kept first, same names/order as the live sheet; extra columns are to the right"),
    ("attendance values", "present / late / absent / excused (the live dashboard rule counts only 'present')"),
    ("month", "billing month = ceil(session_number / 4)"),
    ("student_session_number", "nth record of the student"), ("student_month", "ceil(student_session_number / 4)"),
    ("student_session", "position inside the student's month (1-4)  [ASSUMPTION - confirm against the live bot]"),
    ("percentage", "exam_score / max_exam_score (blank when absent/excused)"),
    ("Payments.status", "paid / paid_late / partial / unpaid / exempt (amount_due, amount_paid, due_date, paid_date, method added)"),
    ("Sessions.session_type", "regular / makeup (makeups fall on a different weekday)"),
    ("Hidden truth", "ground_truth_students.csv is NOT in this workbook (kept out so it can be used to evaluate models)"),
    ("", ""), ("Calibration (actual / target)", ""),
] + [(r.metric, f"{r.actual}   (target {r.target})") for r in calib.itertuples()]

wb = Workbook()
try:
    wb._named_styles["Normal"].font = Font(name="Arial", size=10)
except Exception:
    pass
HDR_FILL = PatternFill("solid", fgColor="252A34"); HDR_FONT = Font(name="Arial", size=10, bold=True, color="EAEAEA")
WIDTHS = {"name": 28, "group_name": 28, "notes": 24, "attendance_notes": 16, "area": 14, "key": 24, "detail": 30}
first_sheet = True
for name, df in TABLES.items():
    ws = wb.active if first_sheet else wb.create_sheet()
    first_sheet = False
    ws.title = name
    cols = list(df.columns)
    ws.append(cols)
    for c in range(1, len(cols) + 1):
        cell = ws.cell(row=1, column=c); cell.fill = HDR_FILL; cell.font = HDR_FONT
    d = df.astype(object).where(df.notna(), None)
    is_date = [c for c in cols if c in ("date", "join_date", "leave_date", "start_date", "due_date", "paid_date")]
    for row in d.itertuples(index=False):
        out = []
        for c, v in zip(cols, row):
            if c == "time" and isinstance(v, str):
                hh = int(v.split(":")[0]); pm = v.endswith("PM")
                v = dt.time((hh % 12) + (12 if pm else 0), 0)
            elif isinstance(v, (np.integer,)): v = int(v)
            elif isinstance(v, (np.floating,)): v = float(v)
            out.append(v)
        ws.append(out)
    for ci, c in enumerate(cols, start=1):
        ws.column_dimensions[get_column_letter(ci)].width = WIDTHS.get(c, 14)
        fmt = "m/d/yyyy" if c in is_date else "0.0%" if c == "percentage" else "h:mm:ss AM/PM" if c == "time" else None
        if fmt:
            for r in range(2, ws.max_row + 1):
                ws.cell(row=r, column=ci).number_format = fmt
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
ws = wb.create_sheet("README")
for k, v in README:
    ws.append([k, v if not isinstance(v, (np.integer, np.floating)) else v.item()])
ws.column_dimensions["A"].width = 52; ws.column_dimensions["B"].width = 95
ws["A1"].font = Font(name="Arial", size=12, bold=True)
wb.save(os.path.join(OUT_DIR, "Students_Management_ML_Dataset.xlsx"))
print("saved ->", os.path.abspath(OUT_DIR))

### Cell 13 — (Optional) Upload to Google Sheets | لو عندك Service Account (مش بيتنفذ تلقائيًا)
**يتحط:** بعد Cell 12

In [ ]:
USE_GSHEETS = False
if USE_GSHEETS:
    import gspread
    gc = gspread.service_account(filename="service_account.json")      # needs a service account shared on the target sheet
    sh = gc.create("Students Management - ML Dataset")                  # NEW file: never write on the live sheet
    for name, df in TABLES.items():
        ws_ = sh.add_worksheet(title=name, rows=len(df) + 1, cols=len(df.columns))
        vals = [list(df.columns)] + df.astype(object).where(df.notna(), "").astype(str).values.tolist()
        for i in range(0, len(vals), 5000):                              # batches of 5000 rows
            ws_.update(f"A{i + 1}", vals[i:i + 5000], value_input_option="USER_ENTERED")